In [1]:
!git clone https://github.com/Aarav-cyber/Business-Entity-Resolution-Challenge.git


Cloning into 'Business-Entity-Resolution-Challenge'...
remote: Enumerating objects: 51, done.
remote: Counting objects: 100% (51/51), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 51 (delta 13), reused 48 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (51/51), 31.31 KiB | 15.65 MiB/s, done.
Resolving deltas: 100% (13/13), done.


In [2]:
%cd Business-Entity-Resolution-Challenge

/content/Business-Entity-Resolution-Challenge


In [3]:
!pip install -q pandas numpy scikit-learn rapidfuzz xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 35.0 MB/s eta 0:00:00


In [4]:
!find src -maxdepth 3 -type f

src/blocking/diagnose_misses.py
src/blocking/blocking.py
src/blocking/evaluate_blocking.py
src/blocking/run_blocking.py
src/data/sample_cleaned (2).tsv
src/data/eda.ipynb
src/data/normalize.py
src/data/sample_cleaned.tsv
src/data/loader.py
src/model/features.py
src/model/infer.py
src/model/train.py
src/eval/split.py
src/eval/evaluate.py


In [5]:
!pip install -q dropbox

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 700.2/700.2 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.0/204.0 kB 12.8 MB/s eta 0:00:00


In [7]:
import dropbox
from google.colab import userdata

try:
    # Securely retrieve the token from Colab Secrets
    DROPBOX_ACCESS_TOKEN = userdata.get('DROPBOX_ACCESS_TOKEN')
except Exception:
    # Fallback to manual entry if not set in Secrets
    DROPBOX_ACCESS_TOKEN = "YOUR_DROPBOX_ACCESS_TOKEN"

if not DROPBOX_ACCESS_TOKEN or DROPBOX_ACCESS_TOKEN == "YOUR_DROPBOX_ACCESS_TOKEN":
    raise ValueError("Please set your 'DROPBOX_ACCESS_TOKEN' in Colab Secrets (the 🔑 icon on the left panel) or paste it directly in the cell.")

dbx = dropbox.Dropbox(DROPBOX_ACCESS_TOKEN)

account = dbx.users_get_current_account()
print("Connected successfully!")
print("Account:", account.name.display_name)

Connected successfully!
Account: Aarav Raj Shrestha


In [8]:
metadata, response = dbx.files_download(
    "/6ab10eb3b23ba_student_resource.zip"
)

zip_path = "/content/student_resource.zip"

with open(zip_path, "wb") as f:
    f.write(response.content)

print("Downloaded to temporary Colab storage.")


Downloaded to temporary Colab storage.


In [9]:
import zipfile
import os

extract_path = "/content/student_resource"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Extracted successfully!")

Extracted successfully!


In [ ]:
import pandas as pd
from pathlib import Path

train_dir = Path("/content/student_resource/student_resource/dataset/train")

for f in train_dir.glob("*.tsv"):
    print(f"{f.name:25} {f.stat().st_size / 1024**2:,.1f} MB")

In [ ]:
for f in train_dir.glob("*.tsv"):
    with open(f, "rb") as file:
        rows = sum(1 for _ in file) - 1
    print(f"{f.name}: {rows:,} rows")

In [ ]:
import pandas as pd
from pathlib import Path

test_dir = Path("/content/student_resource/student_resource/dataset/test")

for f in test_dir.glob("*.tsv"):
    print(f"{f.name:25} {f.stat().st_size / 1024**2:,.1f} MB")

In [ ]:
for f in test_dir.glob("*.tsv"):
    with open(f, "rb") as file:
        rows = sum(1 for _ in file) - 1
    print(f"{f.name}: {rows:,} rows")

In [ ]:
import os

!python src/blocking/run_blocking.py \
    --data-dir /content/student_resource/student_resource/dataset/train \
    --out output/train_candidate_pairs.tsv \
    --max-token-df 5000 \
    --min-token-idf 2.5 \
    --top-k-tfidf 10 \
    --tfidf-max-dist 0.55 \
    --max-candidates 30

In [ ]:
!python /content/Business-Entity-Resolution-Challenge/src/blocking/fast_blocking.py --data-dir /content/student_resource/student_resource/dataset/test --out /tmp/smoke.tsv --sample-s1 500


In [ ]:
!python /content/Business-Entity-Resolution-Challenge/src/blocking/fast_blocking.py --data-dir /content/student_resource/student_resource/dataset/test --out /content/Business-Entity-Resolution-Challenge/output/candidate_pairs.tsv


In [ ]:
import csv
import random

input_file = "/content/student_resource/student_resource/dataset/train/train_ground_truth.tsv"
output_file = "/content/Business-Entity-Resolution-Challenge/src/data/sample_train_100K_ground_truth.tsv"
N = 100000

with open(input_file, "r", encoding="utf-8") as f:
    reader = csv.reader(f, delimiter="\t")

    header = next(reader)
    sample = []

    for i, row in enumerate(reader):
        if i < N:
            sample.append(row)
        else:
            j = random.randint(0, i)
            if j < N:
                sample[j] = row

with open(output_file, "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f, delimiter="\t")
    writer.writerow(header)
    writer.writerows(sample)

print(f"Created {output_file} with {len(sample)} rows")

In [11]:
!python3 /content/Business-Entity-Resolution-Challenge/src/stage_c_full_scale.py calibrate --train-dir /content/student_resource/student_resource/dataset/train --out /content/threshold.json

ground-truth S1 with matches: 2083574, referenced S2/S3 ids: 7638365
usable positive pairs: 7638365  negative samples: 16668592
threshold=0.43 calibration_F0.5=0.976 -> /content/threshold.json


In [16]:
!python3 /content/Business-Entity-Resolution-Challenge/src/stage_c_full_scale_2.py infer \
    --test-dir /content/student_resource/student_resource/dataset/test \
    --candidates /content/Business-Entity-Resolution-Challenge/output/candidate_pairs.tsv \
    --threshold /content/threshold.json \
    --out /content/Business-Entity-Resolution-Challenge/output/matching_results.tsv \
    --db /content/stage_c_index.sqlite

using threshold=0.43
Building on-disk index (bounded RAM, uses disk at --db)...
  indexed 1732544 rows into table 's1'
  indexed 9969589 rows into table 'cand'
rows written: 1732544 (of which 1722953 had no candidate-file row -> empty)
rows with >=1 match: 5268
-> /content/Business-Entity-Resolution-Challenge/output/matching_results.tsv
(you can delete the index db now: /content/stage_c_index.sqlite)
